# RAG jurídico — consecuencias del incumplimiento de un contrato

Proyecto final SI3003 · Opción A (sistema RAG) · **LangChain + NVIDIA NIM + Qdrant**

Este notebook ejecuta el prototipo completo usando el paquete `rag_juridico` del repositorio:

| Paso | Qué hace |
|---|---|
| 1 | Obtener el repositorio e instalar dependencias |
| 2 | Configurar la clave de NVIDIA de forma privada |
| 3 | Ingesta: documentos → fragmentos → embeddings → índice Qdrant |
| 4 | Consultar un caso y ver la respuesta **con sus fuentes** |
| 5 | Caso límite: el sistema reconoce cuando el contexto no alcanza |
| 6 | Ejecutar los cinco casos de prueba |
| 7 | (Opcional) Interfaz Gradio |

## 1. Repositorio y dependencias

En Colab, escribe la URL del repositorio del equipo. En local, abre el notebook desde la carpeta `notebooks/` del proyecto y esta celda solo ubicará la raíz.

In [1]:
import os, sys
from pathlib import Path

URL_REPO = ""  # p. ej. "https://github.com/<usuario>/<repositorio>.git" (solo necesario en Colab)

if not (Path.cwd() / "rag_juridico").is_dir():
    if (Path.cwd().parent / "rag_juridico").is_dir():      # ejecutando desde notebooks/
        os.chdir(Path.cwd().parent)
    elif URL_REPO:                                          # Colab
        destino = Path(URL_REPO.rstrip("/").split("/")[-1].removesuffix(".git"))
        if not destino.exists():
            !git clone -q {URL_REPO}
        os.chdir(destino)
    else:
        raise RuntimeError("Indica URL_REPO o abre el notebook desde la carpeta del proyecto.")

sys.path.insert(0, str(Path.cwd()))
print("Raíz del proyecto:", Path.cwd())

Raíz del proyecto: c:\Users\Lenovo\OneDrive\Escritorio\I.M\Inteligencia Artificial\Proyecto\RAG_Juridico


In [ ]:
%pip install -q -r requirements.txt

## 2. Clave de NVIDIA (privada)

La clave **no se escribe en el notebook**. Se busca en este orden: secreto de Colab `NVIDIA_API_KEY` (🔑 *Secrets*) → variable de entorno o archivo `.env` → entrada oculta por teclado.

In [2]:
from rag_juridico.config import Config
from rag_juridico.proveedores import obtener_api_key

obtener_api_key()
config = Config.desde_entorno()
print("✅ Clave configurada")
print("Embeddings:", config.modelo_embeddings, "| LLM:", config.modelo_generacion)

✅ Clave configurada
Embeddings: nvidia/nemotron-3-embed-1b | LLM: nvidia/nemotron-3.5-lightning-30b-a3b


## 3. Ingesta

Lee los documentos de `data/leyes` y `data/sentencias`, los fragmenta (las leyes **por artículo**), genera embeddings y los guarda en Qdrant (`storage/qdrant`). Solo hay que repetirla cuando cambian los documentos.

In [8]:
from rag_juridico.ingesta import ingestar

manifiesto = ingestar(config)

1/4 Cargando documentos de C:\Users\Lenovo\OneDrive\Escritorio\I.M\Inteligencia Artificial\Proyecto\RAG_Juridico\data ...
  [ley      ] Código Civil colombiano  (618 pág., 1,518,597 caracteres)
  [ley      ] Código de Comercio colombiano - Decreto 410 de 1971  (635 pág., 1,347,497 caracteres)
  [sentencia] Responsabilidad civil contractual  (30 pág., 71,886 caracteres)
  [sentencia] Tribunal Superior de Medellín - Responsabilidad civil contractual  (58 pág., 98,797 caracteres)
  ⚠ sentencias/Responsabilidad_Civil_Contractual.pdf: sin origen declarado en data/fuentes.json.
2/4 Fragmentando ...
  4 documentos -> 5642 fragmentos {'ley': 5460, 'sentencia': 182}
3/4 Generando embeddings (nvidia/nemotron-3-embed-1b) ...
  5642/5642
4/4 Indexando en qdrant (dimensión 2048) ...
✅ Índice listo: 5642 fragmentos de 4 documentos.


In [3]:
# Inspección: así queda un fragmento con sus metadatos
from rag_juridico.ingesta import leer_fragmentos

fragmentos = leer_fragmentos(config.ruta_fragmentos)
ejemplo = fragmentos[len(fragmentos) // 2]
print(ejemplo.encabezado())
print("-" * 80)
print(ejemplo.texto[:600])

Ley | Código Civil colombiano | art. 2466, p. 542
--------------------------------------------------------------------------------
ARTICULO 2466. <COMPENSACION DE FRUTO S CON LOS INTERESES>. Las partes podrán
estipular que los frutos se compensen con los intereses, en su totalidad, o hasta concurrencia de valores.

Los intereses que estipularen estarán sujetos, en caso de lesión enorme, a la misma reducción que en el
caso de mutuo.


## 4. Consultar un caso

`responder()` ejecuta cinco etapas: (1) el LLM identifica los elementos del caso y redacta consultas en lenguaje jurídico; (2) recuperación híbrida (similitud coseno + BM25, fusionadas con RRF) con cupo por tipo de fuente; (3) control de suficiencia; (4) generación restringida al contexto con citas `[F#]`; (5) verificación de las citas.

In [3]:
from IPython.display import Markdown, display
from rag_juridico.pipeline import AsistenteJuridico
from rag_juridico.presentacion import a_markdown

asistente = AsistenteJuridico.desde_storage(config)



c:\Users\Lenovo\anaconda3\lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [5]:
caso = (
    "¿Qué establece el Código Civil sobre la entrega de una cosa vendida?"
)
respuesta = asistente.responder(caso)
display(Markdown(a_markdown(respuesta)))

**Estado:** 🟡 Respondida parcialmente (el contexto cubre solo una parte)

### Hechos relevantes
El usuario consulta qué establece el Código Civil sobre la entrega de una cosa vendida. No se han expuesto hechos concretos de un caso particular; la consulta es general sobre la materia.

### Qué dicen las fuentes
- **[F1] Art. 1880 (Código Civil):** Establece que las obligaciones del vendedor se reducen en general a dos: la entrega o tradición, y el saneamiento de la cosa vendida. La tradición se sujetará a las reglas del título VI del libro II.
- **[F2] Art. 928 (Código de Comercio):** El vendedor está obligado a entregar lo que reza el contrato, con todos sus accesorios, en las mismas condiciones que tenía al momento de perfeccionarse. Si la cosa es un cuerpo cierto, deberá conservarla hasta su entrega bajo pena de indemnizar los perjuicios al comprador, salvo que la pérdida o deterioro se deban a fuerza mayor o caso fortuito (carga de la prueba recae en el vendedor).
- **[F3] Art. 1882 (Código Civil):** El vendedor está obligado a entregar la cosa inmediatamente después del contrato o a la época prefijada. Si el vendedor ha retardado la entrega por hecho o culpa suya, el comprador puede perseverar en el contrato o desistir, con derecho a indemnización de perjuicios. Si después del contrato menguía considerablemente la fortuna del comprador, de modo que el vendedor se hallara en peligro inminente de perder el precio, no se podrá exigir la entrega salvo pagar o asegurar el pago.
- **[F4] Art. 1849 (Código Civil):** Define la compraventa como un contrato en que una parte se obliga a dar una cosa y la otra a pagarla en dinero.
- **[F5] Art. 1884 (Código Civil):** El vendedor está obligado a entregar lo que reza el contrato.
- **[F6] Art. 934 (Código de Comercio):** Si la cosa vendida presenta vicios o defectos ocultos cuya causa sea anterior al contrato, ignorados sin culpa por el comprador, y hagan la cosa impropia para su natural destinación o el fin previsto, el comprador podrá pedir la resolución del contrato o la rebaja del precio. Si opta por la resolución, deberá restituir la cosa. Habrá indemnización de perjuicios si el vendedor conocía o debía conocer el vicio al tiempo del contrato.
- **[F7] Art. 1927 (Código Civil):** Si la compra se hizo para remitir la cosa a lugar distante, la acción de rebaja del precio prescribirá en un año contado desde la entrega al consignatario, más el término de emplazamiento por distancia. Es necesario que el comprador haya podido ignorar el vicio sin negligencia entre la venta y la remesa.
- **[F8] Art. 923 (Código de Comercio):** La entrega se entenderá verificada por diversas modalidades (transmisión de conocimiento de embarque, fijación de marca, expedición al domicilio, u otros medios autorizados). La expedición no implicará entrega si se hizo sin ánimo de transferir la propiedad (ej. remisión a consignatario con orden de no entregar hasta pago).

### Alcance y límites
Los fragmentos recuperados permiten identificar las reglas generales sobre la entrega en el contrato de compraventa según el Código Civil y el Código de Comercio colombiano. Se establece la obligación del vendedor de entregar la cosa en las condiciones pactadas, los efectos del retraso, las modalidades de entrega y las consecuencias ante vicios ocultos.

No obstante, no se pueden extraer conclusiones sobre un caso concreto (por ejemplo, si una entrega específica es válida o si procede una acción por vicio oculto) sin conocer los hechos particulares y la normativa aplicable a la situación fáctica. La consulta original es de carácter general y el análisis se limita a exponer lo que dicen los artículos recuperados.

**⚠ Advertencias de verificación**
- El modelo no declaró la suficiencia. La respuesta se clasificó provisionalmente como parcial.

---
### Elementos identificados en el caso
(no se pudo estructurar el caso; se usa el texto original)

### Fuentes recuperadas
**[F1] Ley · Código Civil colombiano · art. 1880, p. 436** — citada, similitud 0.53
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1880. <OBLIGACIONES DEL VENDEDOR>. Las obligaciones del vendedor se reducen
  > en general a dos: la entrega o tradición, y el saneamiento de la cosa vendida.
  > 
  > La tradición se sujetará a las reglas dadas en el título VI del libro II.
  > 
  > CODIGO CIVIL COLOMBIANO
  > Página 437 de 618

**[F2] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 928, p. 219** — citada, similitud 0.52
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6
  > Art. 928._Obligación de entregar. El vende dor estará obligado a entregar lo que reza el
  > contrato, con todos sus accesorios, en las mismas condiciones que tenía al momento de
  > perfeccionarse; y si la cosa vendida es un cuerpo cierto, estará obligado a conservarla
  > hasta su entrega so pena de indemniz ar los perjuicios al comprador, salvo que la pérdida
  > o deterioro se deban a fuerza mayor o caso fortuito, cuya prueba corresponderá al
  > vendedor.
  > Conc.: 923, 927, 931; C. Civil 1605 a 1608, 1857, 1884.

**[F3] Ley · Código Civil colombiano · art. 1882, p. 437** — citada, similitud 0.50
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1882. <TIEMPO DE ENTREGA Y RETARDO>. El vendedor es obligado a entregar la
  > cosa vendida inmediatamente después del contrato, o a la época prefijada en él.
  > 
  > Si el vendedor, por hecho o culpa suya ha retardado la entrega, podrá el comprador, a su arbitrio,
  > perseverar en el contrato o desistir de é l y en ambos casos con derecho para ser indemnizado de los
  > perjuicios según las reglas generales.
  > 
  > Todo lo cual se entiende si el comprador ha pagado o está pronto a pagar el precio íntegro o ha
  > estipulado pagar a plazo.
  > 
  > Pero si después del contrato hubie re menguado considerablemente la fortuna del comprador, de modo
  > que el vendedor se halle en peligro inminente de perder el precio, no se …

**[F4] Ley · Código Civil colombiano · art. 1849, p. 431** — citada, similitud 0.49
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1849. <CONCEPTO DE COMPRAVENTA>. La compraventa es un contrato en que una
  > de las partes se obliga a dar una cosa y la otra a pa garla en dinero. Aquélla se dice vender y ésta
  > comprar. El dinero que el comprador da por la cosa vendida se llama precio.

**[F5] Ley · Código Civil colombiano · art. 1884, p. 437** — citada, similitud 0.57
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1884. <OBJETO DE LA ENTREGA>. El vendedor es obligado a entregar lo que reza el
  > contrato.

**[F6] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 934, p. 221** — citada, coincidencia léxica
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6
  > Art. 934._Vicios ocultos. Si la cosa vendida presenta, con posterioridad a su entrega
  > vicios o defectos ocultos cuya causa sea anterior al contrato, ignorados sin culpa por el
  > comprador, que hagan la cosa impropia para su natural destinación o para el fin previsto
  > en el contrato, el comprador tendrá derecho a pedir la resolución del mismo o la rebaja del
  > precio a justa tasación. Si el comprador opta por la resolución, deberá restituir la cosa al
  > vendedor.
  > En uno u otro caso habrá lugar a indemnización de perjuicios por parte del vende dor, si
  > éste conocía o debía conocer al tiempo del contrato el vicio o el defecto de la cosa
  > vendida.
  > Conc.: 870, 932, 935, 938, 945; C. Civil 1883 y ss., 1914 …

**[F7] Ley · Código Civil colombiano · art. 1927, p. 443** — citada, coincidencia léxica
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1927. <PRESCRIPCION CUANDO LA COSA VENDIDA IBA A SER ENVIADA A
  > LUGAR DISTANTE>. Si la compra se ha hecho para remitir la cosa a lugar distante, la acción de
  > rebaja del precio prescribirá en un año contado desde la entrega al consignatario, con más el término de
  > emplazamiento que corresponda a la distancia.
  > 
  > Pero será necesario que el comprador, en el tiempo intermedio entre la venta y la remesa, haya podido
  > ignorar el vicio de la cosa, sin negligencia de su parte.
  > 
  > CAPITULO IX.
  > 
  > DE LAS OBLIGACIONES DEL COMPRADOR

**[F8] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 923, p. 218** — citada, similitud 0.52
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6
  > Art. 923._Modalidades de la entrega. La entrega de la cosa se entenderá verificada:
  > 1o) Por la transmisión del conocimiento de embarque, carta de porte o fac tura, durante el
  > transporte de las mercaderías por tierra, mar y aire;
  > 2o) Por la fijación que haga el comprador de su marca en las mercaderías compradas,
  > con conocimiento y aquiescencia del vendedor;
  > 3o) Por la expedición que haga el vendedor de las merca derías al domicilio del comprador
  > o a cualquier otro lugar convenido, sin perjuicio de los dispuesto en el artículo 915.
  > La expedición no implicará entrega cuando sea efectuada sin ánimo de transferir la
  > propiedad, como cuando el vendedor ha remitido las m ercaderías a un consignatari …

_Este prototipo académico identifica y explica información de documentos indexados; no determina responsabilidad contractual ni sustituye asesoría jurídica profesional._

In [5]:
# Qué ocurrió por dentro: consultas generadas y puntajes de los fragmentos recuperados
print("Consultas de búsqueda:")
for c in respuesta.consultas:
    print("  •", c[:150])
print(f"\nMejor similitud: {respuesta.mejor_similitud:.3f} (umbral de suficiencia: {config.umbral_suficiencia})\n")
for f in respuesta.fuentes:
    sim = f"{f.similitud:.3f}" if f.similitud is not None else "léxica"
    print(f"{'✔' if f.citada else ' '} {f.etiqueta}  [{sim}]")

Consultas de búsqueda:
  • Ana compró un apartamento a Luis y pagó la totalidad del precio el día acordado. Han pasado tres meses desde la fecha pactada para la entrega y Luis n

Mejor similitud: 0.422 (umbral de suficiencia: 0.35)

  [F1] Ley · Código Civil colombiano · art. 1882, p. 437  [0.422]
  [F2] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 926, p. 219  [léxica]
  [F3] Ley · Código Civil colombiano · art. 1930, p. 443  [0.386]
  [F4] Sentencia · Tribunal Superior de Medellín - Responsabilidad civil contractual · Antecedentes, p. 2  [0.375]
  [F5] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 944, p. 223  [léxica]
  [F6] Ley · Código Civil colombiano · art. 1984, p. 452  [0.370]
  [F7] Ley · Código Civil colombiano · art. 2002, p. 455  [léxica]
  [F8] Sentencia · Tribunal Superior de Medellín - Responsabilidad civil contractual · Antecedentes, p. 3  [0.359]


## 5. Caso límite: información ausente

El corpus trata contratos civiles y comerciales. Ante una pregunta de otro tema el sistema debe **abstenerse**: si ningún fragmento supera el umbral de similitud no se llama al modelo generador; y si se llama, el prompt le exige declarar `SUFICIENCIA: INSUFICIENTE`.

In [6]:
limite = asistente.responder(
    "Mi empleador me despidió sin justa causa tras cinco años de contrato laboral a término indefinido. "
    "¿Cuántos días de salario me corresponden como indemnización?"
)
print("Estado:", limite.estado, "| mejor similitud:", round(limite.mejor_similitud, 3))
display(Markdown(a_markdown(limite, con_texto=False)))

Estado: insuficiente | mejor similitud: 0.401


**Estado:** 🔴 El modelo declaró que el contexto no es suficiente

El contexto no contiene la regla legal para calcular la indemnización por despido injustificado en un contrato laboral a término indefinido. Los fragmentos disponibles regulan contratos de agencia mercantil, sirvientes por tiempo cierto y casos específicos con montos concretos, pero no establecen el criterio ni el cálculo de días de salario para el escenario planteado. Por tanto, no es posible determinar la cantidad solicitada basándose únicamente en el CONTEXTO proporcionado.

---
### Elementos identificados en el caso
(no se pudo estructurar el caso; se usa el texto original)

### Fuentes recuperadas
**[F1] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 1327, p. 327** — no citada, similitud 0.40
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F2] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 1324, p. 325** — no citada, similitud 0.36
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F3] Sentencia · Responsabilidad civil contractual · p. 2** — no citada, similitud 0.35
  - Archivo: `sentencias/Responsabilidad_Civil_Contractual.pdf`

**[F4] Ley · Código Civil colombiano · art. 2049, p. 464** — no citada, similitud 0.37
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F5] Sentencia · Responsabilidad civil contractual · p. 27** — no citada, similitud 0.38
  - Archivo: `sentencias/Responsabilidad_Civil_Contractual.pdf`

**[F6] Sentencia · Responsabilidad civil contractual · p. 4** — no citada, similitud 0.37
  - Archivo: `sentencias/Responsabilidad_Civil_Contractual.pdf`

**[F7] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 1419, p. 342** — no citada, coincidencia léxica
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F8] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 1659, p. 403** — no citada, coincidencia léxica
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

_Este prototipo académico ofrece información basada en los documentos indexados; no constituye asesoría jurídica profesional._

## 6. Los cinco casos de prueba

Ejecuta `evaluacion/casos_prueba.json` y genera un informe en `evaluacion/resultados/` con pregunta, resultado esperado, respuesta, fuentes y un espacio para la **valoración manual** del equipo.

In [4]:
from rag_juridico.evaluacion import evaluar

informe = evaluar(asistente)
display(Markdown(informe.read_text(encoding="utf-8")))

▶ C1 (Consulta documental — compraventa y entrega) ...
▶ C2 (Consulta documental — arrendamiento y cláusula penal) ...
▶ C3 (Consulta documental — contrato de obra) ...
▶ C4 (Comparación documental — pérdida fortuita de cuerpo cierto) ...
▶ C5 (Fuera de alcance — indemnización laboral) ...
✅ Informe: C:\Users\Lenovo\OneDrive\Escritorio\I.M\Inteligencia Artificial\Proyecto\RAG_Juridico\evaluacion\resultados\resultados_20261114_232824.md


# Resultados de evaluación — 20261114_232824

## C1 — Consulta documental — compraventa y entrega

**Pregunta / caso:** Ana compró un apartamento a Luis, pagó todo el precio y la fecha pactada para la entrega pasó hace tres meses sin que Luis entregara el inmueble. Según los documentos disponibles, ¿qué establece el Código Civil sobre la obligación de entrega del vendedor y sobre las alternativas que menciona cuando hay retraso? No determines si Luis debe indemnizar a Ana en este caso concreto.

**Resultado esperado:** Debe identificar la falta de entrega y explicar fielmente el artículo 1882 del Código Civil: obligación de entregar en el momento pactado y alternativas de perseverar o desistir cuando el retraso sea por hecho o culpa del vendedor, con referencia a perjuicios según esa condición. No debe presumir culpa ni decidir una indemnización específica.

**Estado obtenido:** 🟡 Respondida parcialmente (el contexto cubre solo una parte) (mejor similitud 0.75, 8.0 s)

**Respuesta obtenida:**

### Hechos relevantes
Ana compró un apartamento a Luis, pagó el precio total y la fecha pactada para la entrega venció hace tres meses sin que Luis entregara el inmueble.

### Qué dicen las fuentes
Según el Código Civil colombiano, el vendedor tiene la obligación de entregar la cosa vendida inmediatamente después del contrato o a la época prefijada [F1, art. 1882]. Si el vendedor retrasa la entrega por su hecho o culpa, el comprador tiene dos alternativas a su arbitrio: perseverar en el contrato o desistir de él. En ambos casos, el comprador tiene derecho a ser indemnizado de los perjuicios según las reglas generales [F1, art. 1882]. Además, el vendedor está obligado a entregar lo que reza el contrato [F3, art. 1884].

### Alcance y límites
Los fragmentos recuperados establecen que el retraso en la entrega otorga al comprador la opción de mantener la obligación o rescindir el contrato, más el derecho a indemnización de perjuicios. No se determina en el caso concreto si el retraso fue por hecho o culpa del vendedor, ni si el comprador ejerció alguna de las alternativas legales. Tampoco se identifica si existen cláusulas pactadas que modifiquen estos efectos. Para una decisión sobre la situación concreta se requeriría información sobre la conducta de las partes y eventuales acuerdos específicos.

**Fuentes recuperadas:**

**[F1] Ley · Código Civil colombiano · art. 1882, p. 437** — citada, similitud 0.75
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F2] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 943, p. 222** — no citada, similitud 0.62
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F3] Ley · Código Civil colombiano · art. 1884, p. 437** — citada, similitud 0.70
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F4] Ley · Código Civil colombiano · art. 1880, p. 436** — no citada, similitud 0.70
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F5] Ley · Código Civil colombiano · art. 1984, p. 452** — no citada, similitud 0.60
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F6] Ley · Código Civil colombiano · art. 667, p. 210** — no citada, similitud 0.57
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F7] Ley · Código Civil colombiano · art. 1930, p. 443** — no citada, similitud 0.63
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F8] Sentencia · Tribunal Superior de Medellín - Responsabilidad civil contractual · Antecedentes, p. 10** — no citada, coincidencia léxica
  - Archivo: `sentencias/Sentencia_2022.pdf`
  - Origen: https://tribunalmedellin.com/images/decisiones/civil/2023/05001310301020200008501.pdf

**Comprobaciones automáticas (ayuda, no reemplazan la revisión manual):**
- ¿Estado esperado? sí (esperado: respondida, parcial)
- Tipos de fuente esperados que faltaron: ninguno
- Términos esperados ausentes: ninguno
- Advertencias de citas: El modelo no declaró la suficiencia. La respuesta se clasificó provisionalmente como parcial.

**Valoración manual del equipo:** _(¿es relevante?, ¿cada afirmación está respaldada por el fragmento citado?, ¿coincide con el documento original?)_

---

## C2 — Consulta documental — arrendamiento y cláusula penal

**Pregunta / caso:** En un arrendamiento de local comercial, el arrendatario dejó de pagar tres cánones y el contrato contempla una cláusula penal de dos cánones. ¿Qué dicen los fragmentos recuperados sobre la falta de pago de la renta y sobre la cláusula penal? Identifica los artículos y sus condiciones, sin calcular cuánto se puede cobrar ni afirmar que la penalidad es válida en este contrato.

**Resultado esperado:** Debe describir, si se recuperan, el artículo 2035 del Código Civil sobre la falta de pago de renta y las condiciones que menciona, y el artículo 867 del Código de Comercio sobre cláusula penal. Debe distinguir entre describir esas reglas y decidir su aplicación. No debe afirmar que se pueden cobrar exactamente cinco cánones ni asegurar la validez o acumulación de la pena.

**Estado obtenido:** 🟡 Respondida parcialmente (el contexto cubre solo una parte) (mejor similitud 0.52, 11.4 s)

**Respuesta obtenida:**

### Hechos relevantes
El arrendatario de un local comercial dejó de pagar tres cánones (períodos de renta) y el contrato incluye una cláusula penal estipulada en dos cánones.

### Qué dicen las fuentes

**Sobre la falta de pago de la renta:**
- **[F3] Art. 2035**: Establece que la mora de un período entero en el pago de la renta, después de dos reconvenciones (mínimo cuatro días entre ellas), da derecho al arrendador a cesar el arrendamiento si no se presta seguridad de pago dentro de un plazo razonable (mínimo treinta días).
- **[F7] Art. 2013**: Señala que, si el arrendamiento cesa por desahucio o vencimiento de duración, el arrendatario debe pagar la renta de todos los días que faltan para que cese, aunque haya restituido voluntariamente el local antes del último día.
- **[F6] Art. 2003**: Indica que, cuando el arrendamiento se termina por culpa del arrendatario, este debe indemnizar perjuicios y pagar la renta por el tiempo que faltaba hasta que, desahucio mediante, el arrendador pudiera cesar el contrato. El arrendatario puede eximitarse proponiendo un sustituto idóneo y prestando fianza.

**Sobre la cláusula penal:**
- **[F1] Art. 949 (Código de Comercio)** y **[F2] Art. 1592 (Código Civil)**: Definen la cláusula penal como aquel pacto por el cual una persona se sujeta a una pena (entregar o hacer algo) en caso de no ejecutar o retardar la obligación principal. El Art. 949 conecta esta figura con el Art. 867 del Código Civil.
- **[F5] Art. 1593**: Establece que la nulidad de la obligación principal acarrea la de la cláusula penal, pero la nulidad de la penal no extingue la obligación principal. Además, la pena sigue siendo exigible aunque la obligación principal carezca de efecto por falta de consentimiento de la persona prometida, o cuando la estipulación es a favor de un tercero.

### Alcance y límites
Los fragmentos recuperados permiten identificar las reglas sobre la mora en el pago de la renta (requisitos para cesar el arrendamiento) y la definición y efectos de la cláusula penal (vinculación a la obligación principal y consecuencias de su nulidad). No obstante, no se encuentran en el corpus disposiciones que regulen específicamente los límites, la procedencia o el cálculo de la penalidad estipulada en cánones en contratos de arrendamiento comercial, ni se definen las condiciones particulares de aplicación de la cláusula penal de dos cánones en el caso de falta de pago de tres cánones. Para un análisis sobre la validez o el monto de la penalidad en este contrato concreto, se requeriría consultar el texto completo del contrato y, potencialmente, normas adicionales no recuperadas en este contexto.

**Fuentes recuperadas:**

**[F1] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 949, p. 223** — citada, similitud 0.48
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F2] Ley · Código Civil colombiano · art. 1592, p. 377** — citada, similitud 0.45
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F3] Ley · Código Civil colombiano · art. 2035, p. 461** — citada, similitud 0.43
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F4] Ley · Código Civil colombiano · art. 2486, p. 544** — no citada, similitud 0.46
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F5] Ley · Código Civil colombiano · art. 1593, p. 377** — citada, similitud 0.40
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F6] Ley · Código Civil colombiano · art. 2003, p. 455** — citada, similitud 0.39
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F7] Ley · Código Civil colombiano · art. 2013, p. 457** — citada, similitud 0.40
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F8] Sentencia · Tribunal Superior de Medellín - Responsabilidad civil contractual · Consideraciones, p. 22** — no citada, coincidencia léxica
  - Archivo: `sentencias/Sentencia_2022.pdf`
  - Origen: https://tribunalmedellin.com/images/decisiones/civil/2023/05001310301020200008501.pdf

**Comprobaciones automáticas (ayuda, no reemplazan la revisión manual):**
- ¿Estado esperado? sí (esperado: respondida, parcial)
- Tipos de fuente esperados que faltaron: ninguno
- Términos esperados ausentes: ninguno
- Advertencias de citas: El modelo no declaró la suficiencia. La respuesta se clasificó provisionalmente como parcial.

**Valoración manual del equipo:** _(¿es relevante?, ¿cada afirmación está respaldada por el fragmento citado?, ¿coincide con el documento original?)_

---

## C3 — Consulta documental — contrato de obra

**Pregunta / caso:** Una empresa contrató una remodelación y debía pagar un anticipo del 30 % antes de iniciar. No pagó el anticipo y el contratista no empezó la obra. ¿Qué disposiciones recuperadas tratan el incumplimiento y los perjuicios en los contratos de obra? Explica lo que dicen, pero no decidas si una demanda de alguna de las partes prosperaría.

**Resultado esperado:** Debe reconocer los hechos del anticipo y explicar el artículo 2056 del Código Civil, si está recuperado, sobre la obra material y la reclamación de perjuicios según las reglas generales. No debe convertir la facultad de quien encarga la obra para hacerla cesar en un supuesto derecho del contratista a suspenderla. Si no se recupera una norma sobre obligaciones recíprocas, debe decirlo sin inventar esa regla.

**Estado obtenido:** 🟡 Respondida parcialmente (el contexto cubre solo una parte) (mejor similitud 0.59, 11.4 s)

**Respuesta obtenida:**

### Hechos relevantes
Una empresa (comitente) contrató una remodelación y acordó pagar un anticipo del 30 % antes de iniciar la obra. La empresa no realizó ese pago y, como consecuencia, el contratista no comenzó los trabajos.

### Qué dicen las fuentes
- **[F6]** Art. 1552: "Lo que se paga antes de cumplirse el plazo, no está sujeto a restitución. Esta regla no se aplica a los plazos que tienen el valor de condiciones." Este fragmento establece la regla general sobre pagos anticipados, indicando que, salvo que el plazo tenga valor de condición, el pago anticipado no genera derecho a devolución.

- **[F2]** Art. 2056: "Habrá lugar a reclamación de perjuicios, según las reglas generales de los contratos, siempre que por una o por otra parte no se ha ya ejecutado lo convenido, o se haya retardado su ejecución." Este artículo permite reclamar perjuicios cuando no se ejecuta lo convenido. Además, señala que "el que encargó la obra, aún en el caso de haberse estipulado un precio único y total por ella, podrá hacerla cesar, reembolsando al artífice todos los costos, y dándole lo que valga el trabajo hecho, y lo que hubiera podido ganar en la obra". Esto implica que, si el comitente no paga el anticipo y el contratista no inicia, el contratista podría cesar en la ejecución y reclamar por el trabajo no realizado y los lucros cesantes.

- **[F5]** Art. 1616: "Si no se puede imputar dolo al deudor, solo es responsable de los perjuicios que se previeron o pudieron preverse al tiempo del contrato; pero si hay dolo, es responsable de todos los perjuicios que fueron consecuencia inmediata o directa de no haberse cumplido la obligación o de haberse demorado su cumplimiento. La mora producida por fuerza mayor o caso fortuito, no da lugar a indemnización de perjuicios." Este fragmento delimita la responsabilidad por perjuicios: solo se indemnizan los previstos o previsibles, salvo que haya dolo. La mora por fuerza mayor o caso fortuito exime de indemnización.

- **[F7]** Art. 1614: Define "daño emergente" (perjuicio o pérdida por no cumplimiento) y "lucro cesante" (ganancia que se deja de reportar). Estas figuras son la base para calcular los perjuicios en caso de incumplimiento.

- **[F1]** Art. 2062: Trata la resolución del contrato por muerte del artífice o empresario, y la obligación de recibir trabajos o materiales preparados. No es directamente aplicable al caso de no pago de anticipo, pero forma parte del capítulo sobre contratos de construcción.

### Alcance y límites
Los fragmentos recuperados permiten explicar las reglas generales sobre pagos anticipados ([F6]), la facultad de cesar la obra y reclamar perjuicios por falta de ejecución ([F2]), y los conceptos de daño emergente y lucro cesante ([F7]), así como el límite de responsabilidad por perjuicios previstos frente al dolo ([F5]). No se dispone en el corpus de una norma específica que regule el anticipo del 30 % ni de una cláusula penal o resolutoria pactada por las partes. Para determinar el régimen exacto aplicable a este caso (si el anticipo es exigible, si su falta permite la resolución del contrato y qué perjuicios se pueden reclamar), sería necesario conocer el contenido exacto de la cláusula de anticipo y si las partes estipularon penalidades o condiciones resolutorias.

**Fuentes recuperadas:**

**[F1] Ley · Código Civil colombiano · art. 2062, p. 467** — citada, similitud 0.51
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F2] Ley · Código Civil colombiano · art. 2056, p. 465** — citada, similitud 0.59
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F3] Ley · Código Civil colombiano · art. 2059, p. 466** — no citada, similitud 0.57
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F4] Ley · Código Civil colombiano · art. 2052, p. 465** — no citada, coincidencia léxica
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F5] Ley · Código Civil colombiano · art. 1616, p. 382** — citada, similitud 0.52
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F6] Ley · Código Civil colombiano · art. 1552, p. 371** — citada, similitud 0.50
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F7] Ley · Código Civil colombiano · art. 1614, p. 382** — citada, similitud 0.48
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F8] Sentencia · Responsabilidad civil contractual · p. 5** — no citada, coincidencia léxica
  - Archivo: `sentencias/Responsabilidad_Civil_Contractual.pdf`

**Comprobaciones automáticas (ayuda, no reemplazan la revisión manual):**
- ¿Estado esperado? sí (esperado: respondida, parcial)
- Tipos de fuente esperados que faltaron: ninguno
- Términos esperados ausentes: ninguno
- Advertencias de citas: El modelo no declaró la suficiencia. La respuesta se clasificó provisionalmente como parcial.

**Valoración manual del equipo:** _(¿es relevante?, ¿cada afirmación está respaldada por el fragmento citado?, ¿coincide con el documento original?)_

---

## C4 — Comparación documental — pérdida fortuita de cuerpo cierto

**Pregunta / caso:** Un caballo individualmente determinado fue vendido y murió por un rayo antes de su entrega, sin culpa ni mora del vendedor. El comprador ya había pagado. ¿Qué diferencias aparecen entre el artículo 1876 del Código Civil y el artículo 929 del Código de Comercio sobre quién soporta el riesgo? ¿Qué dice el artículo 1729 sobre la extinción de la obligación? Limítate a comparar el texto recuperado; no decidas qué régimen aplica ni si debe devolverse el precio.

**Resultado esperado:** Debe contrastar expresamente el artículo 1876 del Código Civil (riesgo a cargo del comprador como regla descrita) con el 929 del Código de Comercio (riesgo a cargo del vendedor antes de la entrega, con excepciones). Si se recupera el artículo 1729, debe describir la extinción de la obligación de entregar por pérdida del cuerpo cierto y sus salvedades. No debe concluir definitivamente sobre la devolución del precio, la mora del comprador o el régimen aplicable.

**Estado obtenido:** 🟡 Respondida parcialmente (el contexto cubre solo una parte) (mejor similitud 0.63, 9.2 s)

**Respuesta obtenida:**

### Hechos relevantes
Se consulta sobre un caso de compraventa de un caballo individualmente determinado que murió por rayo antes de la entrega, tras haber sido pagado el precio por el comprador. La pregunta se centra en comparar el artículo 1876 del Código Civil, el artículo 929 del Código de Comercio y el artículo 1729 del Código Civil.

### Qué dicen las fuentes

**Artículo 1876 del Código Civil [F2]:** Establece que la pérdida, deterioro o mejora de la cosa cierta pertenece al comprador desde la perfección del contrato, aunque no se haya entregado la cosa. La excepción es cuando la venta es bajo condición suspensiva: si perece la cosa mientras pende la condición, la pérdida recae en el vendedor.

**Artículo 929 del Código de Comercio [F1]:** En la venta de un "cuerpo cierto", el riesgo de pérdida por fuerza mayor o caso fortuito antes de la entrega corresponde al vendedor, salvo que el comprador se constituya en mora de recibirlo y que la fuerza mayor o el caso fortuito no lo hubiera destruido sin la mora del comprador. En ese caso, el comprador debe el precio íntegro.

**Artículo 1729 del Código Civil [F3]:** Cuando el cuerpo cierto que se debe perece (por destrucción, al dejar de estar en el comercio o al desaparecer), se extingue la obligación, salvo las excepciones de los artículos subsiguientes.

### Alcance y límites
- El artículo 1876 (Código Civil) sitúa el riesgo en el comprador desde la perfección del contrato, sin distinguir expresamente la causa de la pérdida (fuerza mayor o caso fortuito) en el fragmento recuperado.
- El artículo 929 (Código de Comercio) atribuye el riesgo al vendedor antes de la entrega, pero con la salvedad de que, si el comprador está en mora y la cosa se destruye sin que ello dependa de su mora, el comprador debe el precio.
- El artículo 1729 (Código Civil) regla la extinción de la obligación por pérdida de la cosa, pero menciona que salvan las excepciones de los artículos subsiguientes, lo que indica que la regla no es absoluta.
- No se determina el régimen aplicable al caso concreto, ya que el análisis requiere contrastar si el contrato se rige por el Código Civil o el Código de Comercio, y si la condición suspensiva o la mora del comprador están presentes.

**Fuentes recuperadas:**

**[F1] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 929, p. 219** — citada, similitud 0.63
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F2] Ley · Código Civil colombiano · art. 1876, p. 436** — citada, similitud 0.63
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F3] Ley · Código Civil colombiano · art. 1729, p. 402** — citada, similitud 0.61
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F4] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 2038, p. 572** — no citada, similitud 0.48
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F5] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 952, p. 224** — no citada, similitud 0.41
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F6] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 928, p. 219** — no citada, similitud 0.45
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F7] Ley · Código Civil colombiano · art. 1899, p. 439** — no citada, coincidencia léxica
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F8] Ley · Código Civil colombiano · art. 1607, p. 380** — no citada, similitud 0.43
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**Comprobaciones automáticas (ayuda, no reemplazan la revisión manual):**
- ¿Estado esperado? sí (esperado: respondida, parcial)
- Tipos de fuente esperados que faltaron: ninguno
- Términos esperados ausentes: ninguno
- Advertencias de citas: ninguna

**Valoración manual del equipo:** _(¿es relevante?, ¿cada afirmación está respaldada por el fragmento citado?, ¿coincide con el documento original?)_

---

## C5 — Fuera de alcance — indemnización laboral

**Pregunta / caso:** Trabajo hace cinco años con contrato laboral a término indefinido y mi empleador me despidió sin justa causa. ¿Cuántos días de salario me corresponden como indemnización por despido?

**Resultado esperado:** Debe clasificar la consulta como fuera del alcance documental y abstenerse brevemente de calcular días o citar reglas laborales no recuperadas. No debe dar fórmulas, cifras, artículos del Código Sustantivo del Trabajo no aportados por el corpus ni conclusiones sobre el derecho del usuario.

**Estado obtenido:** 🔴 El modelo declaró que el contexto no es suficiente (mejor similitud 0.00, 2.8 s)

**Respuesta obtenida:**

Esta consulta corresponde al derecho laboral, una materia fuera del alcance de este prototipo de consulta documental sobre contratos civiles y comerciales. No es posible responderla adecuadamente con el corpus disponible.

**Fuentes recuperadas:**

_No se recuperaron fragmentos suficientemente relacionados._

**Comprobaciones automáticas (ayuda, no reemplazan la revisión manual):**
- ¿Estado esperado? sí (esperado: sin_contexto, insuficiente)
- Tipos de fuente esperados que faltaron: ninguno
- Términos esperados ausentes: ninguno
- Advertencias de citas: ninguna

**Valoración manual del equipo:** _(¿es relevante?, ¿cada afirmación está respaldada por el fragmento citado?, ¿coincide con el documento original?)_

---


## 7. (Opcional) Interfaz Gradio

Cierra antes el asistente de este notebook: Qdrant en modo local solo admite un proceso a la vez sobre la misma carpeta.

In [ ]:
asistente.indice.cerrar()
!RAG_GRADIO_SHARE=1 python app.py

In [14]:
respuesta_normal = asistente.responder(caso)

print("Estado:", respuesta_normal.estado)
print("Longitud:", len(respuesta_normal.respuesta))
print("Advertencias:", respuesta_normal.advertencias)

Estado: respondida
Longitud: 2814
Advertencias: []


In [4]:
from rag_juridico.proveedores import crear_llm

asistente.llm = crear_llm(config).bind(
    chat_template_kwargs={
        "enable_thinking": False
    }
)

In [6]:
from IPython.display import Markdown, display
from rag_juridico.presentacion import a_markdown

display(Markdown(a_markdown(respuesta)))

**Estado:** 🟡 Respondida parcialmente (el contexto cubre solo una parte)

### 1. Elementos del caso
- **Tipo de contrato:** Compraventa inmobiliaria.
- **Partes:** Ana (compradora); Luis (vendedor).
- **Obligaciones:** Ana pagó el precio total; Luis debía entregar el inmueble en la fecha pactada.
- **Incumplimiento:** Luis no entregó el inmueble tres meses después de la fecha pactada.
- **Cumplimiento de Ana:** Ana cumplió su obligación de pago.

### 2. Normas aplicables
- **[F1] Art. 1882 Código Civil:** Establece que si el vendedor retrasa la entrega, el comprador puede perseverar en el contrato o desistir de él, con derecho a ser indemnizado de los perjuicios. Esto es aplicable si el comprador ha pagado el precio íntegro.
- **[F2] Art. 942 Código de Comercio:** En caso de resolución por incumplimiento del vendedor, el comprador tiene derecho al pago del interés legal comercial sobre la parte pagada del precio o a retener los frutos en proporción a dicha parte, sin menoscabo de la indemnización de perjuicios.

### 3. Jurisprudencia y casos relacionados
- **[F7] Art. 1932 Código Civil:** Regula los efectos de la resolución por no pago del precio, estableciendo que el comprador tiene derecho a la restitución de la parte pagada del precio, y el vendedor a retener las arras o exigirlas dobladas, más la restitución de frutos. Aunque el caso es de falta de entrega y no de no pago, la lógica de la restitución de lo pagado es análoga.
- **[F8] Sentencia Tribunal Superior de Medellín:** Menciona la caracterización de contratos de compraventa y la transferencia de inmuebles, sirviendo como referencia sobre la naturaleza de los contratos objeto del análisis, aunque no detalla las consecuencias específicas del retraso en la entrega en este fragmento.

### 4. Consecuencias del incumplimiento
- **Derecho a la resolución:** Basado en **[F1]**, Ana tiene el arbitrio de perseverar en el contrato o desistir de él (resolución).
- **Indemnización de perjuicios:** En ambos casos (perseverancia o resolución), Ana tiene derecho a ser indemnizada de los perjuicios según las reglas generales ([F1]).
- **Interés legal o frutos:** Si Ana opta por la resolución, **[F2** le asiste el derecho a que se le pague el interés legal comercial sobre la parte pagada del precio o a retener los frutos de la cosa en proporción a dicha parte.
- **Restitución de lo pagado:** Si el contrato se resuelve, Ana tendría derecho a la restitución de la parte del precio que pagó, tal como se desprende de la lógica de **[F7]** aplicada al comprador (restitución de lo pagado).

### 5. Lo que el contexto no permite afirmar
El contexto no permite determinar si el incumplimiento es imputable a Luis o si existe una causa externa (fuerza mayor) que exonere su responsabilidad. Además, no se especifica si el contrato contiene una cláusula penal o penalidad específica por retraso que modifique el cálculo de los intereses o indemnizaciones; el contexto solo ofrece las reglas generales de indemnización y resolución.

### Conclusión
Dado que Ana ha pagado el precio total y Luis no ha entregado el inmueble tres meses después del plazo pactado, Ana tiene la facultad de elegir entre perseverar en el contrato y exigir la entrega inmediata más una indemnización por los daños y perjuicios causados por el retraso, o resolver el contrato y exigir la restitución de la totalidad del precio pagado, más el interés legal comercial sobre dicha parte o el retención de los frutos proporcionales, sin perjuicio de la indemnización de perjuicios por el retraso.

---
### Elementos identificados en el caso
- Tipo de contrato: compraventa
- Partes: Ana (compradora); Luis (vendedor)
- Obligaciones: Ana: pagar el precio total. Luis: entregar el inmueble en la fecha pactada y en las condiciones acordadas
- Incumplimiento: Luis no ha entregado el inmueble tres meses después de la fecha pactada, incumpliendo su obligación principal de entrega. Ana cumplió su obligación de pago.
- Causa: no_determinada
- Cláusulas relevantes: cláusula de plazo, posible cláusula penal, garantía de dominio
- Pregunta jurídica: Qué acciones legales puede ejercer el comprador contra el vendedor por incumplimiento del contrato de compraventa inmobiliaria tras el pago del precio y la falta de entrega.
- Datos que faltan: Si el contrato establece una cláusula penal o penalidad por retraso, y si existen condiciones resolutorias tácitas o expresas.

### Fuentes recuperadas
**[F1] Ley · Código Civil colombiano · art. 1882, p. 437** — citada, similitud 0.52
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1882. <TIEMPO DE ENTREGA Y RETARDO>. El vendedor es obligado a entregar la
  > cosa vendida inmediatamente después del contrato, o a la época prefijada en él.
  > 
  > Si el vendedor, por hecho o culpa suya ha retardado la entrega, podrá el comprador, a su arbitrio,
  > perseverar en el contrato o desistir de é l y en ambos casos con derecho para ser indemnizado de los
  > perjuicios según las reglas generales.
  > 
  > Todo lo cual se entiende si el comprador ha pagado o está pronto a pagar el precio íntegro o ha
  > estipulado pagar a plazo.
  > 
  > Pero si después del contrato hubie re menguado considerablemente la fortuna del comprador, de modo
  > que el vendedor se halle en peligro inminente de perder el precio, no se …

**[F2] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 942, p. 222** — citada, similitud 0.50
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6
  > Art. 942._Resolución de contrato por incumplimiento de vendedor. Derechos del
  > comprador. En caso de resol ución de una compraventa por incumplimiento del vendedor,
  > el comprador tendrá derecho a que se le pague el interés legal comercial sobre la parte
  > pagada del precio o a retener los frutos de la cosa en proporción a dicha parte, sin
  > menoscabo de la correspon diente indemnización de perjuicios.
  > Conc.: 883, 870, 884, 962; C. Civil 1613 a 1617, 1882.
  > 
  > CAPITULO V
  > OBLIGACIONES DEL COMPRADOR

**[F3] Ley · Código Civil colombiano · art. 1930, p. 443** — no citada, similitud 0.54
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1930. <MORA EN EL PAGO DEL PRECIO>. Si el comprador estuvier e constituido en
  > mora de pagar el precio en el lugar y tiempo dichos, el vendedor tendrá derecho para exigir el precio o
  > la resolución de la venta, con resarcimiento de perjuicios.

**[F4] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 950, p. 224** — no citada, similitud 0.48
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6
  > Art. 950._Derechos del vendedor. En caso de incumplimiento del comprador, el
  > vendedor tendrá derecho a una justa retribución por el uso que el comprador haya hecho
  > de la cosa y a la restitución de los frutos en proporción a la parte no pagada del precio, sin
  > menoscabo de la correspondiente indemnización de perjuicios.
  > Conc.: 919; C. Civil 714 a 718, 1932; C. de P. C. 427 par. 2o. No. 12, 435 par. 2º.
  > 
  > CAPITULO VI
  > VENTA CON PLAZO PARA EL PAGO DEL PRECIO
  > Y CON RESERVA DEL DOMINIO

**[F5] Ley · Código Civil colombiano · art. 1983, p. 451** — no citada, similitud 0.39
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1983. <DESISTIMIENTO DEL CONTRATO POR INCUMPLIMIENTO EN LA
  > ENTREGA>. Si el arrendador, por hecho o culpa suya o de sus agentes o dependientes, se ha puesto en
  > la imposibilidad de entregar la cosa, el arrendatario tendrá derecho para desistir del contrato, con
  > indemnización de perjuicios.
  > 
  > Habrá lugar a esta indemnización aún cuando el arrendador haya creído erróneamente y de buena fe
  > que podía arrendar la cosa; salvo que la imposibilidad haya sido conocida del arr endatario, o provenga
  > de fuerza mayor o caso fortuito.
  > 
  > CODIGO CIVIL COLOMBIANO
  > Página 452 de 618

**[F6] Ley · Código Civil colombiano · art. 1984, p. 452** — no citada, similitud 0.49
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1984. <MORA EN LA ENTREGA>. Si el arrendador, por hecho o culpa suya o de sus
  > agentes o dependientes, es constituido en mora de entregar, tendrá derecho el arrendatario a
  > indemnización de per juicio.
  > 
  > Si por el retardo se disminuyere notablemente para el arrendatario la utilidad del contrato, sea por
  > haberse deteriorado la cosa o por haber cesado las circunstancias que lo motivaron, podrá el
  > arrendatario desistir del contrato, quedándole a salv o la indemnización de perjuicios, siempre que el
  > retardo no provenga de fuerza mayor o caso fortuito.

**[F7] Ley · Código Civil colombiano · art. 1932, p. 444** — citada, similitud 0.50
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1932. <EFECTOS DE LA RESOLUCION POR NO PAGO>. La resolución de la venta
  > por no haberse pagado el precio dará derecho al vendedor para retener las arras, o exigirlas dobladas, y
  > además para que se le restituyan los frutos, ya en su totalidad si ninguna parte del precio se le hubiere
  > pagado, ya en la proporción que corresponda a la parte del precio que no hubiere sido pagada.
  > 
  > El comp rador, a su vez, tendrá derecho para que se le restituya la parte que hubiere pagado del precio.
  > 
  > Para el abono de las expensas al comprador, y de los deterioros al vendedor, se considerará al primero
  > como poseedor de mala fe, a menos que pruebe haber sufr ido en su fortuna, y sin culpa de su parte,
  > meno …

**[F8] Sentencia · Tribunal Superior de Medellín - Responsabilidad civil contractual · Consideraciones, p. 45** — citada, similitud 0.40
  - Archivo: `sentencias/Sentencia_2022.pdf`
  - Origen: https://tribunalmedellin.com/images/decisiones/civil/2023/05001310301020200008501.pdf
  > 7. Caso concreto.
  > 
  > 7.1. Caracterización de las negociaciones atacadas.
  > 
  > Para efectos de claridad, debe recordarse que los
  > contratos de compraventa objeto del escrutinio de la Corte
  > son solamente los siguientes:
  > 
  > n.° E.P. FECHA Vendedor Comprador Inmuebles
  > Enajenados
  > 1 629 01/09/2010 Carlos Enrique
  > Correcha Jiménez Carlos Arturo
  > Correcha Oliveros 368-9275, 368 -
  > 9276 y 368 -9277 2 748 06/10/2010 Carlos Arturo
  > Correcha Oliveros Inversiones Likarla
  > S.A.
  > 3 625 01/09/2010 Carlos Enrique
  > Correcha Jiménez Luz Esther
  > Correcha 368-38641
  > 4 749 06/10/2010 Luz Esther
  > Correcha Inversiones Likarla
  > S.A.
  > 5 626 01/09/2010 Lida Esperanza
  > Díaz Cabrera Luz Esther
  > Correcha 368-41189
  > 6 747 06/10/2010 Luz Esther
  > C …

_Este prototipo académico ofrece información basada en los documentos indexados; no constituye asesoría jurídica profesional._